# Regional profile: reported, attributable, and mutation-specific

Project-mean bars for `rq2_pattern_trend_comparison.png`.

Each bar is the unweighted mean of the per-project percentages. A region with no categorized mutants (NA) is left out of that mean. The denominator of every percentage is the categorized mutants in that project-region pair.

- **Reported** is the published regional table. Fluct. is S→K→S + K→S→K + recurrent, summed inside each project-region before the mean.
- **Attributable** and **Mutation-specific** are computed from `mu_region_all.csv`. Attributable mutants are those listed in `mu_region.csv`. Mutation-specific is the subset with `mutation_specific = yes`.

Attributable and Mutation-specific share one y-axis. Reported keeps its own scale.

In [ ]:
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.transforms import Bbox
import numpy as np

In [ ]:
REGIONS = ["r1", "r2", "r3", "r4"]
PATTERNS = ["S→K", "K→S", "Fluct."]

HERE = Path.cwd()
if not (HERE / "data" / "mu_region_all.csv").exists():
    for cand in [HERE, *HERE.parents]:
        if (cand / "data" / "mu_region_all.csv").exists() and (cand / "rq2_region_assign.py").exists():
            HERE = cand
            break
DATA = HERE / "data"
df = pd.read_csv(DATA / "mu_region_all.csv", low_memory=False)
df = df[df.table_region.isin(REGIONS)].copy()
for col in ("attributable", "mutation_specific"):
    df[col] = df[col].astype(str).str.lower().eq("true")


def project_means(flag):
    """Unweighted mean across projects of 100 * flagged mutants with the pattern / categorized."""
    means = {pat: [] for pat in PATTERNS}
    for region in REGIONS:
        per_project = defaultdict(list)
        for project, group in df[df.table_region == region].groupby("project"):
            denom = len(group)
            if denom == 0:
                continue
            for pat in PATTERNS:
                num = ((group.pattern == pat) & group[flag]).sum()
                per_project[pat].append(100.0 * num / denom)
        for pat in PATTERNS:
            vals = per_project[pat]
            means[pat].append(sum(vals) / len(vals))
    return means


attributable = project_means("attributable")
mutation_specific = project_means("mutation_specific")
pd.DataFrame({"region": REGIONS, **{f"attr {k}": v for k, v in attributable.items()}})

Published percentages by project and region, in order r1–r4. `None` is NA.

In [ ]:
REPORTED = {
    "black": {
        "S→K": [0.0, 0.6, 0.4, 0.2],
        "K→S": [0.1, 0.0, 0.0, 0.0],
        "S→K→S": [0.0, 0.0, 0.0, 0.0],
        "K→S→K": [10.5, 5.8, 5.3, 0.0],
        "Rec.": [0.7, 0.2, 0.6, 0.0],
    },
    "cookiecutter": {
        "S→K": [0.0, 0.0, 5.9, 0.0],
        "K→S": [0.0, 0.0, 0.0, 0.0],
        "S→K→S": [0.0, 0.0, 0.0, 0.0],
        "K→S→K": [0.0, 0.0, 0.0, 0.0],
        "Rec.": [0.0, 0.0, 0.0, 0.0],
    },
    "fastapi": {
        "S→K": [0.0, 3.3, 1.6, 1.8],
        "K→S": [0.0, 0.0, 0.0, 0.2],
        "S→K→S": [0.0, 0.0, 0.0, 0.0],
        "K→S→K": [0.0, 0.0, 0.0, 0.0],
        "Rec.": [0.0, 0.0, 0.0, 0.0],
    },
    "httpie": {
        "S→K": [38.1, 35.8, 1.4, 7.0],
        "K→S": [0.0, 0.0, 0.0, 0.0],
        "S→K→S": [0.0, 0.0, 0.0, 0.0],
        "K→S→K": [0.0, 0.0, 0.0, 0.0],
        "Rec.": [0.0, 0.0, 0.0, 0.0],
    },
    "sanic": {
        "S→K": [0.0, None, 5.3, None],
        "K→S": [0.0, None, 1.3, None],
        "S→K→S": [0.0, None, 3.9, None],
        "K→S→K": [0.0, None, 0.0, None],
        "Rec.": [0.0, None, 13.2, None],
    },
    "thefuck": {
        "S→K": [4.9, 1.4, 1.4, 2.2],
        "K→S": [6.3, 2.4, 2.1, 0.9],
        "S→K→S": [0.6, 0.5, 0.9, 3.6],
        "K→S→K": [0.0, 0.0, 0.5, 0.1],
        "Rec.": [0.2, 0.0, 0.3, 0.4],
    },
    "tqdm": {
        "S→K": [4.3, 11.6, 4.1, 4.5],
        "K→S": [1.0, 0.8, 3.6, 7.1],
        "S→K→S": [1.6, 2.1, 5.8, 1.5],
        "K→S→K": [2.3, 0.9, 0.2, 1.6],
        "Rec.": [15.8, 26.6, 23.7, 14.9],
    },
    "youtube-dl": {
        "S→K": [1.2, 0.4, 0.1, None],
        "K→S": [2.0, 1.5, 0.0, None],
        "S→K→S": [4.8, 0.0, 2.2, None],
        "K→S→K": [0.0, 0.0, 0.0, None],
        "Rec.": [0.0, 0.0, 0.0, None],
    },
}


def reported_means():
    means = {pat: [] for pat in PATTERNS}
    for i in range(4):
        buckets = {pat: [] for pat in PATTERNS}
        for project, cells in REPORTED.items():
            if cells["S→K"][i] is None:
                continue
            buckets["S→K"].append(cells["S→K"][i])
            buckets["K→S"].append(cells["K→S"][i])
            buckets["Fluct."].append(cells["S→K→S"][i] + cells["K→S→K"][i] + cells["Rec."][i])
        for pat in PATTERNS:
            vals = buckets[pat]
            means[pat].append(sum(vals) / len(vals))
    return means


reported = reported_means()
pd.DataFrame({"region": REGIONS, **reported}).round(2)

In [ ]:
def padded(bb):
    return Bbox.from_extents(bb.x0 - 3, bb.y0 - 2, bb.x1 + 3, bb.y1 + 2)


def draw(panels, path):
    colors = ["#4C78A8", "#F58518", "#54A24B"]
    x = np.arange(len(REGIONS))
    width = 0.28
    region_labels = ["$r_1$", "$r_2$", "$r_3$", "$r_4$"]

    fig, axes = plt.subplots(1, 3, figsize=(16.8, 6.0))
    stored = []
    for ax, (title, series, fmt, shared) in zip(axes, panels):
        ymax = max(max(v) for v in series.values())
        annots = []
        for i, name in enumerate(PATTERNS):
            vals = series[name]
            offset = (i - 1) * width
            ax.bar(x + offset, vals, width * 0.88, label=name, color=colors[i], zorder=3)
            for xpos, value in zip(x + offset, vals):
                annots.append(ax.annotate(
                    fmt.format(value),
                    xy=(xpos, value),
                    xytext=(0, 6),
                    textcoords="offset points",
                    ha="center",
                    va="bottom",
                    fontsize=13,
                    color="#222",
                    zorder=4,
                ))
        ax.set_title(title, fontsize=18, pad=36)
        ax.set_xticks(x)
        ax.set_xticklabels(region_labels, fontsize=17)
        ax.tick_params(axis="y", labelsize=13)
        ax.set_ylabel("Project mean (%)", fontsize=14)
        ax.set_xlim(-0.6, 3.6)
        ax.set_ylim(0, 3.2 if shared else ymax * 1.38)
        ax.yaxis.grid(True, linestyle="-", linewidth=0.7, color="#c8c8c8", zorder=0)
        ax.set_axisbelow(True)
        ax.legend(
            frameon=False,
            fontsize=16,
            ncol=3,
            loc="lower center",
            bbox_to_anchor=(0.5, 1.02),
            borderaxespad=0,
        )
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        stored.append((ax, annots, shared))

    fig.tight_layout()

    def resolve():
        for ax, annots, shared in stored:
            for g in range(4):
                placed = []
                for ann in annots[g::4]:
                    for _ in range(6):
                        fig.canvas.draw()
                        renderer = fig.canvas.get_renderer()
                        bb = padded(ann.get_window_extent(renderer))
                        if not any(bb.overlaps(p) for p in placed):
                            placed.append(bb)
                            break
                        dx, dy = ann.xyann
                        ann.xyann = (dx, dy + 16)
                    else:
                        placed.append(padded(ann.get_window_extent(fig.canvas.get_renderer())))

    resolve()
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    ceiling = 3.2
    for ax, annots, shared in stored:
        if not shared:
            continue
        ax_bb = ax.get_window_extent(renderer)
        top = max(ann.get_window_extent(renderer).y1 for ann in annots)
        overflow = max(0.0, top - ax_bb.y1 + 10)
        ceiling = max(ceiling, 3.2 + overflow * (3.2 / ax_bb.height))
    for ax, annots, shared in stored:
        if shared:
            ax.set_ylim(0, ceiling)
    if ceiling > 3.2:
        resolve()

    fig.savefig(path, dpi=160, bbox_inches="tight")
    return fig


panels = [
    ("Reported", reported, "{:.1f}", False),
    ("Attributable", attributable, "{:.2f}", True),
    ("Mutation-specific", mutation_specific, "{:.2f}", True),
]
draw(panels, str(HERE / "rq2_pattern_trend_comparison.png"))